# Stage 1 — Scaled Dot-Product Attention

Implementing the core attention mechanism from *An Image is Worth 16x16 Words* (Dosovitskiy et al., 2020).

Every token simultaneously plays three roles:
- **Query** (`Q`): *What am I looking for?*
- **Key** (`K`): *What do I advertise about myself?*
- **Value** (`V`): *What do I actually contribute?*

The mechanism scores every query against every key, normalises with softmax, then takes
a weighted sum of values:

```
Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V
```

In [ ]:
import torch
import torch.nn as nn

## 1. Scaled Dot-Product Attention

Given Q, K, V each of shape `[N, d_k]` (N tokens, d_k-dimensional vectors):

1. **Score**: `Q @ K.T` — one score per (query, key) pair → `[N, N]`
2. **Scale**: divide by `sqrt(d_k)` — prevents softmax saturation (see section 2)
3. **Normalize**: `softmax(dim=-1)` — each row becomes a probability distribution
4. **Aggregate**: `weights @ V` — weighted blend of values → `[N, d_v]`

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]

    # [*, N, d_k] @ [*, d_k, N] -> [*, N, N]  <- one score per (query, key) pair
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

    # row i sums to 1 — token i's attention distribution over all positions
    weights = torch.softmax(scores, dim=-1)  # [*, N, N]

    # output[i] is a weighted blend of all value vectors
    return weights @ V  # [*, N, d_v]

In [ ]:
N, d_k = 4, 8
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

out = scaled_dot_product_attention(Q, K, V)
print(out.shape)  # [4, 8] — same sequence length and depth as V

## 2. Why Scale by √d_k?

Each element of `Q @ K.T` is a sum of `d_k` products of random variables.
Its variance grows linearly with `d_k`, so scores get large as the model dimension increases.

Large scores → softmax saturates (one entry → 1.0, rest → 0) → gradient of softmax → 0 → learning stalls.

Dividing by `sqrt(d_k)` keeps the variance at ~1 regardless of model size.

In [ ]:
Q8,  K8  = torch.randn(4, 8),  torch.randn(4, 8)
Q64, K64 = torch.randn(4, 64), torch.randn(4, 64)

print(f"d_k=8  | score std: {(Q8  @ K8.T).std():.2f}")
print(f"d_k=64 | score std: {(Q64 @ K64.T).std():.2f}")
print(f"d_k=64 | after /sqrt(64): {(Q64 @ K64.T / 64**0.5).std():.2f}")

## 3. Single-Head Self-Attention

Q, K, V are not given — they are **learned projections** of the same input X:

```
Q = X @ W_Q    [B, N, d_model] -> [B, N, d_k]
K = X @ W_K
V = X @ W_V
```

`nn.Linear(d_model, d_k, bias=False)` stores W as shape `[d_k, d_model]` and computes
`X @ W.T` — equivalent to the matrix multiply above, with W updated by the optimizer.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)

    def forward(self, X):
        # X: [B, N, d_model]
        Q = self.W_Q(X)  # [B, N, d_k]
        K = self.W_K(X)  # [B, N, d_k]
        V = self.W_V(X)  # [B, N, d_k]
        return scaled_dot_product_attention(Q, K, V)  # [B, N, d_k]

In [ ]:
B, N, d_model, d_k = 2, 4, 16, 8
X = torch.randn(B, N, d_model)

attn = SelfAttention(d_model, d_k)
out  = attn(X)
print(out.shape)  # [2, 4, 8]

## 4. Multi-Head Attention

A single head produces one weighted blend and can only capture one type of relationship.
Multi-head attention runs `h` independent heads in **parallel**, each specialising in a
different dependency type (local, syntactic, long-range, etc.):

```
output = concat(head_1, ..., head_h) @ W_O
```

Convention: `d_k = d_model // h`, so the concatenated output stays at `d_model`.

**Reshape trick**: project to full `d_model` once, then split into heads by reshaping:
```
[B, N, d_model] -> [B, N, h, d_k] -> [B, h, N, d_k]
```
This runs as a single batched matmul rather than `h` separate projections.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # project to full d_model; split into heads after
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        B, N, D = X.shape

        Q = self.W_Q(X)  # [B, N, D]
        K = self.W_K(X)  # [B, N, D]
        V = self.W_V(X)  # [B, N, D]

        # [B, N, D] -> [B, N, num_heads, d_k] -> [B, num_heads, N, d_k]
        Q = Q.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)

        # attention runs independently per head: [B, num_heads, N, d_k]
        out = scaled_dot_product_attention(Q, K, V)

        # [B, num_heads, N, d_k] -> [B, N, D]
        out = out.transpose(1, 2).reshape(B, N, D)

        return self.W_O(out)  # [B, N, D]

In [ ]:
B, N, d_model, num_heads = 2, 4, 256, 8
X = torch.randn(B, N, d_model)

mha = MultiHeadAttention(d_model, num_heads)
out = mha(X)
print(out.shape)  # [2, 4, 256]